In [1]:
from langchain.tools import tool

In [2]:
import requests

In [ ]:

@tool
def get_weather(city:str)-> str:
    """ Get weather from a city and according to the degree create ur own condition """
    data = requests.get(f"https://wttr.in/{city}?format=j1").json()
    temp = data["current_condition"][0]["temp_C"]
    
    return f"{city}: {temp}C"
    
    
    

In [4]:
get_weather("Gingee")

'Gingee: 28C'

In [28]:
from langchain.agents import create_agent


weather_agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    system_prompt="you are a weather agent , who tell weather for the reagions",
    tools=[get_weather]
)

In [29]:
weather_agent.invoke({"messages":"hi who are you"})

{'messages': [HumanMessage(content='hi who are you', additional_kwargs={}, response_metadata={}, id='63ada5b5-bfc6-485d-937d-af2cb4d5f764'),
  AIMessage(content='Hello! I’m your friendly weather assistant. I can give you up‑to‑date forecasts, current conditions, and any weather‑related details you need for cities and regions around the world. Just let me know the place you’re interested in, and I’ll fetch the latest weather for you!', additional_kwargs={'reasoning_content': 'The user says "hi who are you". The system says I\'m a weather agent who tells weather for the regions. So I should respond as a weather agent, introduce myself, ask for location perhaps. I can also mention I can provide weather info. No function call needed.'}, response_metadata={'token_usage': {'completion_tokens': 125, 'prompt_tokens': 149, 'total_tokens': 274, 'completion_time': 0.266924943, 'completion_tokens_details': {'reasoning_tokens': 56}, 'prompt_time': 0.040412206, 'prompt_tokens_details': None, 'queue_

In [30]:
response = weather_agent.invoke({"messages":"hi what is the weather in chennai"})

In [31]:
response["messages"][-1].content

'The current temperature in Chennai is **30\u202f°C**. That’s warm and typical for this time of year, with generally clear skies and moderate humidity. It’s a good day for outdoor activities, but stay hydrated and consider wearing light, breathable clothing. If you’ll be out in the sun for a while, don’t forget sunscreen and a hat.'

In [32]:
response

{'messages': [HumanMessage(content='hi what is the weather in chennai', additional_kwargs={}, response_metadata={}, id='668cea12-fa07-498b-9180-686d45cbf149'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "hi what is the weather in chennai". As a weather agent, we need to fetch weather data using the provided function get_weather, passing city "Chennai". Then we need to respond with a weather description based on degree, creating our own condition. So we should call the function.', 'tool_calls': [{'id': 'fc_75cf5ac7-44d2-40da-8163-78f370cd6132', 'function': {'arguments': '{"city":"Chennai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 153, 'total_tokens': 245, 'completion_time': 0.19284993, 'completion_tokens_details': {'reasoning_tokens': 64}, 'prompt_time': 0.007873146, 'prompt_tokens_details': None, 'queue_time': 0.39610837, 'total_time': 0.200723076}, 'model_name': 'o

# structured output

In [38]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


In [39]:
class weather_response(BaseModel):
    city: str = Field(description="user asked city")
    temp: str = Field(description="temperature of the city")
    condition: str = Field(description="what is the condition of the city like hot,cool,warm")
    suggestion: str = Field(description="what you will suggest for this temperature")
    
    

In [40]:
structured_weather_agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    system_prompt="you are a weather agent, who tell weather for the reagions",
    tools=[get_weather],
    response_format=ToolStrategy(weather_response)
)

In [47]:
response = structured_weather_agent.invoke({"messages":"hi what is the weather in Kashmeer"})


In [48]:
response['structured_response']

weather_response(city='Kashmir', temp='19°C', condition='cool', suggestion='A light jacket or sweater should keep you comfortable.')

# Memory(short term)

In [ ]:
from langgraph.checkpoint import sqlite
from sqlite3 import connect


# pip install langgraph-checkpoint-sqlite = installed for db

In [ ]:
conn = connect("memory.db",check_same_thread=False)

# databse name = memory.db

In [52]:
checkpointer=sqlite.SqliteSaver(conn)

In [53]:
mem_agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    system_prompt="you are a weather agent, who tell weather for the reagions",
    tools=[get_weather],
    checkpointer=checkpointer
)

In [55]:
config={"configurable":{"thread_id":"u1"}}

In [58]:
res = mem_agent.invoke({"messages":"hi i am jeeva"},config=config)

In [59]:
res["messages"][-1].content

'Nice to meet you, Jeeva! 😊  \n\nJust tell me the city or region you’d like the weather for, and I’ll pull up the latest forecast for you. 🌤️'

In [60]:
res = mem_agent.invoke({"messages":"what is my name"},config=config)


In [61]:
res["messages"][-1].content

'Your name is Jeeva. 🌟'